In [3]:
import sys

from matplotlib import pyplot as plt
%reload_ext autoreload
%autoreload 2

from tqdm.auto import tqdm

from mcfacts.inputs.settings_manager import SettingsManager
from mcfacts.modules.accretion import ProgradeBlackHoleAccretion, BinaryBlackHoleAccretion, ProgradeBlackHoleBondi
from mcfacts.modules.damping import ProgradeBlackHoleDamping, BinaryBlackHoleDamping
from mcfacts.modules.disk_capture import EvolveRetrogradeBlackHoles, RecaptureBinaryBlackHoles, \
    CaptureNSCProgradeBlackHoles
from mcfacts.modules.dynamics import SingleBlackHoleDynamics, BinaryBlackHoleDynamics, BinaryBlackHoleIonization, \
    BinaryBlackHoleEccDynamics, BinaryBlackHoleSpheroidDynamics
from mcfacts.modules.formation import BinaryBlackHoleFormation
from mcfacts.modules.gas_hardening import BinaryBlackHoleGasHardening
from mcfacts.modules.gw import BinaryBlackHoleEvolveGW, InnerBlackHoleDynamics
from mcfacts.modules.merge import ProcessBinaryBlackHoleMergers, ProcessEMRIMergers
from mcfacts.modules.migration import ProgradeBlackHoleMigration, BinaryBlackHoleMigration
from mcfacts.objects.actors import InitialBlackHoleReclassification, InnerDiskFilter, FlipRetroProFilter, InitialStarReclassification
from mcfacts.objects.actors.reality_checks import SingleBlackHoleRealityCheck, BinaryBlackHoleRealityCheck
from mcfacts.objects.agn_object_array import *
from mcfacts.objects.galaxy import Galaxy
from mcfacts.objects.populators import SingleBlackHolePopulator, SingleStarPopulator
from mcfacts.objects.snapshot import TxtSnapshotHandler, IniSnapshotHandler
from mcfacts.objects.timeline import SimulationTimeline
from mcfacts.objects.disk import AGNDisk

# Seed should be 128 bits long to ensure random has enough entropy (as per NumPy documentation)
#
# import secrets
# secret.randombits(128)

settings = SettingsManager({
    "verbose": False,
    "override_files": True,
    "save_state": True,
    "save_each_timestep": True,
    "disk_inner_stable_circ_orb": 16,
    "galaxy_num": 100,
    "output_dir": '../ExampleRuns/NewGTRuns/newseed',
    "active_timestep_num":100,
    "overwrite_files":True,
    "flag_use_pagn": True,
    "seed":54868812644729562268856706214161065189
})

population_cabinet = FilingCabinet()

agn_disk = AGNDisk(settings)
# Create the IO handlers and save the current settings
snapshot_handler = TxtSnapshotHandler(settings = settings)

ini_handler = IniSnapshotHandler(settings=settings)
ini_handler.save_settings("./runs", "settings", settings)

# Load disk model and setup empty filing cabinet for result populations
agn_disk = AGNDisk(settings)
population_cabinet = FilingCabinet()

pbar = tqdm(total=settings.galaxy_num, position=0, leave=True)

for galaxy_id in range(settings.galaxy_num):
    pbar.set_description(f"Running Galaxy {galaxy_id}")
    pbar.update(1)

    # The Galaxy class creates a random generated based on this seed,
    # Philox output for n and n+-1 seeds are uncorrelated
    galaxy_seed = settings.seed - galaxy_id

    # Create instance of galaxy
    galaxy = Galaxy(seed=galaxy_seed, runs_folder="./runs", galaxy_id=str(galaxy_id), settings=settings)

    # Create instance of populators
    single_bh_populator = SingleBlackHolePopulator()
    single_star_populator = SingleStarPopulator()
    galaxy.populate([single_bh_populator, single_star_populator], agn_disk)

    # Create timeline to classify objects created during population
    pre_timeline = SimulationTimeline("Reclassification", timesteps=1, timestep_length=0)

    # Run stars reclassification first, since it can convert stars to bh under certain conditions
    pre_timeline.add_timeline_actor(InitialStarReclassification())
    pre_timeline.add_timeline_actor(InitialBlackHoleReclassification())
    pre_timeline.add_timeline_actor(SingleBlackHoleRealityCheck())

    galaxy.run(pre_timeline, agn_disk)

    # Create timeline to run main simulation
    active_phase_timeline = SimulationTimeline("Active Timeline",
                                           timesteps=settings.active_timestep_num,
                                           timestep_length=galaxy.settings.active_timestep_duration_yr)

    # Initial check to make sure our single black holes are real
    active_phase_timeline.add_timeline_actor(SingleBlackHoleRealityCheck())

    # Get names of different singleton arrays we run through the same module
    # Single retrograde evolution is currently handled separately and not passed as a target array.git
    prograde_array = galaxy.settings.bh_prograde_array_name
    innerdisk_array = galaxy.settings.bh_inner_disk_array_name
    inner_gw_only_array = galaxy.settings.bh_inner_gw_array_name

    # Single Object Physics
    active_phase_timeline.add_timeline_actors([
        ProgradeBlackHoleMigration(target_array=innerdisk_array),
        ProgradeBlackHoleMigration(target_array=prograde_array),
        SingleBlackHoleRealityCheck(),

        ProgradeBlackHoleAccretion(target_array=innerdisk_array),
        ProgradeBlackHoleAccretion(target_array=prograde_array),
        ProgradeBlackHoleDamping(target_array=innerdisk_array),
        ProgradeBlackHoleDamping(target_array=prograde_array),

        EvolveRetrogradeBlackHoles(),
        SingleBlackHoleRealityCheck(),

        InnerBlackHoleDynamics(target_array=innerdisk_array),
        InnerBlackHoleDynamics(target_array=inner_gw_only_array),
        SingleBlackHoleDynamics(target_array=innerdisk_array),
        SingleBlackHoleDynamics(target_array=prograde_array),
    ])

    # Binary Object Physics
    active_phase_timeline.add_timeline_actors([
        BinaryBlackHoleDamping(),

        BinaryBlackHoleDynamics(reality_merge_checks=False),
        ProcessBinaryBlackHoleMergers(),

        BinaryBlackHoleEccDynamics(reality_merge_checks=False),
        ProcessBinaryBlackHoleMergers(),

        BinaryBlackHoleGasHardening(reality_merge_checks=False),
        ProcessBinaryBlackHoleMergers(),

        BinaryBlackHoleAccretion(reality_merge_checks=False),
        ProcessBinaryBlackHoleMergers(),

        BinaryBlackHoleSpheroidDynamics(reality_merge_checks=False),
        ProcessBinaryBlackHoleMergers(),

        RecaptureBinaryBlackHoles(),
        BinaryBlackHoleMigration(),
        BinaryBlackHoleRealityCheck(),

        BinaryBlackHoleEvolveGW(),

        BinaryBlackHoleIonization(),
        ProcessBinaryBlackHoleMergers(),

        BinaryBlackHoleFormation()
    ])

    # Create new prograde black holes
    active_phase_timeline.add_timeline_actor(CaptureNSCProgradeBlackHoles())

    # Population Filters
    active_phase_timeline.add_timeline_actor(InnerDiskFilter())
    active_phase_timeline.add_timeline_actor(FlipRetroProFilter())

    # Handle EMRI Dynamics
    active_phase_timeline.add_timeline_actor(ProcessEMRIMergers())

    # Run the active timeline
    galaxy.run(active_phase_timeline, agn_disk)
 
    #setting new smbh edd to change
    settings = SettingsManager({"smbh_eddington_ratio": 0.5})

    #rebuild the disk with new settings
    agn_disk = AGNDisk(settings)

    #running again
    galaxy.run(active_phase_timeline, agn_disk)

    # Ignore consistency checks on these arrays since they are allowed to have duplicates
    population_cabinet.ignore_consistency_check("blackholes_merged")
    population_cabinet.ignore_consistency_check("blackholes_lvk")

    # Grab array names from settings manager
    bbh_merged_array = galaxy.settings.bbh_merged_array_name
    bbh_lvk_array = galaxy.settings.bbh_gw_array_name
    emri_merged_array = galaxy.settings.emri_array_name
    bh_ejected_array = galaxy.settings.bh_ejected_array_name

    # Sort objects into the final population cabinet containing results from all galaxies
    if bh_ejected_array in galaxy.filing_cabinet:
        population_cabinet.create_or_append_array("blackholes_ejected", galaxy.filing_cabinet.get_array(bh_ejected_array))

    if bbh_merged_array in galaxy.filing_cabinet:
        population_cabinet.create_or_append_array("blackholes_merged", galaxy.filing_cabinet.get_array(bbh_merged_array))

    if bbh_lvk_array in galaxy.filing_cabinet:
        population_cabinet.create_or_append_array("blackholes_lvk", galaxy.filing_cabinet.get_array(bbh_lvk_array))

    if innerdisk_array in galaxy.filing_cabinet:
        population_cabinet.create_or_append_array("blackholes_emri", galaxy.filing_cabinet.get_array(innerdisk_array))

    if inner_gw_only_array in galaxy.filing_cabinet:
        population_cabinet.create_or_append_array("blackholes_emri", galaxy.filing_cabinet.get_array(inner_gw_only_array))

    if emri_merged_array in galaxy.filing_cabinet:
        population_cabinet.create_or_append_array("blackholes_emri", galaxy.filing_cabinet.get_array(emri_merged_array))

pbar.close()

/opt/anaconda3/envs/mcfacts/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3823: UserWarning: Inputted settings dictionary contains override_files, which does not exist in the default settings.Custom settings must be added through the add_custom_settings method.
  exec(code_obj, self.user_global_ns, self.user_ns)
Running Galaxy 1:   2%|▏         | 2/100 [00:05<04:37,  2.83s/it]/Users/vprit/McFACTS/src/mcfacts/modules/dynamics.py:3598: RuntimeWarning: divide by zero encountered in divide
  enc_rate[(bin_orb_a_all > crit_radius)] = 0.002 * (nsc_spheroid_normalization / 0.1) * (
Running Galaxy 3:   4%|▍         | 4/100 [00:11<04:21,  2.73s/it]/Users/vprit/McFACTS/src/mcfacts/modules/dynamics.py:3598: RuntimeWarning: divide by zero encountered in divide
  enc_rate[(bin_orb_a_all > crit_radius)] = 0.002 * (nsc_spheroid_normalization / 0.1) * (
Running Galaxy 4:   5%|▌         | 5/100 [00:13<04:20,  2.75s/it]/Users/vprit/McFACTS/src/mcfacts/modules/dynamics.py:3598: RuntimeWar

In [ ]:
snapshot_handler = TxtSnapshotHandler(settings)

snapshot_handler.save_cabinet("./runs", "population", population_cabinet)

In [ ]:
import matplotlib.ticker as mticker
import numpy as np
from scripts.population_plots import linefunc
from scipy.optimize import curve_fit

from mcfacts.vis import styles
from mcfacts.vis import plotting

figsize = "apj_col"

def make_gen_masks(gen_obj1, gen_obj2):
    """Create masks for retrieving different sets of a merged or binary population based on generation.
    """
    # Column of generation data

    # Masks for hierarchical generations
    # g1 : all 1g-1g objects
    # g2 : 2g-1g and 2g-2g objects
    # g3 : >=3g-Ng (first object at least 3rd gen; second object any gen)
    # Pipe operator (|) = logical OR. (&)= logical AND.
    g1_mask = (gen_obj1 == 1) & (gen_obj2 == 1)
    g2_mask = ((gen_obj1 == 2) | (gen_obj2 == 2)) & ((gen_obj1 <= 2) & (gen_obj2 <= 2))
    gX_mask = (gen_obj1 >= 3) | (gen_obj2 >= 3)

    return g1_mask, g2_mask, gX_mask

mergers = population_cabinet.get_array("blackholes_merged")
mass_final = mergers.mass_final

merger_g1_mask, merger_g2_mask, merger_gX_mask = make_gen_masks(mergers.gen_1, mergers.gen_2)

In [ ]:
# retrieve component masses and mass ratio
m1 = np.zeros(len(mergers))
m2 = np.zeros(len(mergers))
mass_ratio = np.zeros(len(mergers))

for i in range(len(mergers)):
    if mergers.mass_1[i] < mergers.mass_2[i]:
        m1[i] = mergers.mass_1[i]
        m2[i] = mergers.mass_2[i]
    else:
        m1[i] = mergers.mass_2[i]
        m2[i] = mergers.mass_1[i]

    mass_ratio[i] = m1[i] / m2[i]

# (q,X_eff) Figure details here:
# Want to highlight higher generation mergers on this plot
chi_eff = mergers.chi_eff

# Get 1g-1g population
gen1_chi_eff = chi_eff[merger_g1_mask]
gen1_mass_ratio = mass_ratio[merger_g1_mask]
# 2g-1g and 2g-2g population
gen2_chi_eff = chi_eff[merger_g2_mask]
gen_mass_ratio = mass_ratio[merger_g2_mask]
# >=3g-Ng population (i.e., N=1,2,3,4,...)
genX_chi_eff = chi_eff[merger_gX_mask]
genX_mass_ratio = mass_ratio[merger_gX_mask]
# all 2+g mergers; H = hierarchical
genH_chi_eff = chi_eff[(merger_g2_mask + merger_gX_mask)]
genH_mass_ratio = mass_ratio[(merger_g2_mask + merger_gX_mask)]

# points for plotting line fit
x = np.linspace(-1, 1, num=2)

# fit the hierarchical mergers (any binaries with 2+g) to a line passing through 0,1
# popt contains the model parameters, pcov the covariances
# poptHigh, pcovHigh = curve_fit(linefunc, high_gen_mass_ratio, high_gen_chi_eff)

# plot the 1g-1g population
fig = plt.figure(figsize=(plotting.set_size(figsize)[0], 2.8))
ax2 = fig.add_subplot(111)
# 1g-1g mergers
ax2.scatter(gen1_chi_eff, gen1_mass_ratio,
            s=styles.markersize_gen1,
            marker=styles.marker_gen1,
            edgecolor=styles.color_gen1,
            facecolor='none',
            alpha=styles.markeralpha_gen1,
            label='1g-1g'
            )

# plot the 2g+ mergers
ax2.scatter(gen2_chi_eff, gen_mass_ratio,
            s=styles.markersize_gen2,
            marker=styles.marker_gen2,
            edgecolor=styles.color_gen2,
            facecolor='none',
            alpha=styles.markeralpha_gen2,
            label='2g-1g or 2g-2g'
            )

# plot the 3g+ mergers
ax2.scatter(genX_chi_eff, genX_mass_ratio,
            s=styles.markersize_genX,
            marker=styles.marker_genX,
            edgecolor=styles.color_genX,
            facecolor='none',
            alpha=styles.markeralpha_genX,
            label=r'$\geq$3g-Ng'
            )

if len(genH_chi_eff) > 0:
    poptHier, pcovHier = curve_fit(linefunc, genH_mass_ratio, genH_chi_eff)
    errHier = np.sqrt(np.diag(pcovHier))[0]
    # plot the line fitting the hierarchical mergers
    ax2.plot(linefunc(x, *poptHier), x,
             ls='dashed',
             lw=1,
             color='gray',
             zorder=3,
             label=r'$d\chi/dq(\geq$2g)=' +
                   f'{poptHier[0]:.2f}' +
                   r'$\pm$' + f'{errHier:.2f}'
             )
    #         #  alpha=linealpha,

if len(chi_eff) > 0:
    poptAll, pcovAll = curve_fit(linefunc, mass_ratio, chi_eff)
    errAll = np.sqrt(np.diag(pcovAll))[0]
    ax2.plot(linefunc(x, *poptAll), x,
             ls='solid',
             lw=1,
             color='black',
             zorder=3,
             label=r'$d\chi/dq$(all)=' +
                   f'{poptAll[0]:.2f}' +
                   r'$\pm$' + f'{errAll:.2f}'
             )

ax2.set(
    ylabel=r'$q = M_2 / M_1$',  # ($M_1 > M_2$)')
    xlabel=r'$\chi_{\rm eff}$',
    ylim=(0, 1),
    xlim=(-1, 1),
    axisbelow=True
)

if figsize == 'apj_col':
    ax2.legend(loc='lower left', fontsize=6)
elif figsize == 'apj_page':
    ax2.legend(loc='lower left')

ax2.grid('on', color='gray', ls='dotted')